10_xgboost.ipynb
Step 10 — XGBoost Model


XGBoost is an ensemble learning algorithm based on boosting.
It combines multiple weak learners to improve fake news detection accuracy.

10.1 Import Required Libraries
Short Description

Import libraries required for XGBoost model training and evaluation.

In [1]:
# Install XGBoost
!pip install xgboost

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

from xgboost import XGBClassifier

10.2 Load Processed Dataset


Load cleaned and preprocessed fake news dataset.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
file_path = '/content/drive/MyDrive/processed_fake_news_dataset.csv'

df = pd.read_csv(file_path)

10.3 Create Input and Output Variables


Create feature column and target label.

In [4]:
# INPUT FEATURE

X = df["processed_text"]

# TARGET LABEL

y = df["label"]

print("Input and Output Variables Created!")

Input and Output Variables Created!


10.4 TF-IDF Vectorization


Convert text into numerical TF-IDF vectors.

In [5]:
# CREATE TF-IDF

tfidf = TfidfVectorizer(

    max_features=5000
)

# TRANSFORM TEXT

X = tfidf.fit_transform(X)

print("TF-IDF Transformation Completed!")

TF-IDF Transformation Completed!


10.5 Train-Test Split


Split dataset into training and testing sets.

In [6]:
# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42
)

print("Train-Test Split Completed!")

Train-Test Split Completed!


10.6 Create XGBoost Model


Initialize XGBoost ensemble classifier.

In [7]:
# CREATE XGBOOST MODEL

xgb_model = XGBClassifier(

    use_label_encoder=False,

    eval_metric='logloss'
)

print("XGBoost Model Created Successfully!")

XGBoost Model Created Successfully!


10.7 Train XGBoost Model


Train XGBoost model using training data.

In [8]:
# TRAIN MODEL

xgb_model.fit(

    X_train,

    y_train
)

print("XGBoost Model Trained Successfully!")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:18:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Model Trained Successfully!


10.8 Predict Using XGBoost


Predict fake or real news using testing data.

In [9]:
# PREDICTION

xgb_pred = xgb_model.predict(

    X_test
)

print("Prediction Completed Successfully!")

Prediction Completed Successfully!


10.9 Accuracy Evaluation

Calculate model accuracy.

In [10]:
# CALCULATE ACCURACY

xgb_accuracy = accuracy_score(

    y_test,

    xgb_pred
)

print("\nXGBoost Accuracy:")

print(xgb_accuracy)


XGBoost Accuracy:
0.9997762112565738


10.10 Classification Report


Display Precision, Recall and F1-score.

In [11]:
# CLASSIFICATION REPORT

print("\nClassification Report:\n")

print(

    classification_report(

        y_test,

        xgb_pred
    )
)


Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4682
           1       1.00      1.00      1.00      4255

    accuracy                           1.00      8937
   macro avg       1.00      1.00      1.00      8937
weighted avg       1.00      1.00      1.00      8937



10.11 Single News Prediction


Predict custom news as fake or real.

In [12]:
# SAMPLE NEWS

sample_news = [

    "Government launches new education policy"
]

# TF-IDF TRANSFORMATION

sample_vector = tfidf.transform(

    sample_news
)

# PREDICTION

prediction = xgb_model.predict(

    sample_vector
)

# DISPLAY RESULT

if prediction[0] == 1:

    print("\nREAL NEWS")

else:

    print("\nFAKE NEWS")


FAKE NEWS


10.12 Confidence Score Output


Display prediction confidence score.

In [13]:
# CONFIDENCE SCORE

confidence = xgb_model.predict_proba(

    sample_vector
)

print("\nConfidence Score:")

print(confidence)


Confidence Score:
[[9.999685e-01 3.145184e-05]]


10.13 Save XGBoost Model


Save trained model for deployment.

In [ ]:
# ==========================================================
# XGBOOST PIPELINE WITH NLP PREPROCESSING
# ==========================================================

# IMPORTS
import pandas as pd
import numpy as np
import nltk
import string
import joblib

from google.colab import files

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

# ==========================================================
# DOWNLOAD NLTK DATA
# ==========================================================

nltk.download("stopwords")
nltk.download("wordnet")

# ==========================================================
# NLP SETUP
# ==========================================================

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# ==========================================================
# PREPROCESS FUNCTION
# ==========================================================

def preprocess_text(text):

    text = str(text).lower()

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    words = text.split()

    cleaned_words = []

    for word in words:

        if word.isalpha() and word not in stop_words:

            word = lemmatizer.lemmatize(word)
            cleaned_words.append(word)

    return " ".join(cleaned_words)

# ==========================================================
# NLP TRANSFORMER
# ==========================================================

class NLPPreprocessor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        return [
            preprocess_text(text)
            for text in X
        ]

# ==========================================================
# RAW DATA (IMPORTANT FIX)
# ==========================================================

X_raw = df["content"]   # MUST be raw text column
y = df["label"]

# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train_raw, X_test_raw, y_train, y_test = train_test_split(

    X_raw,
    y,

    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train-Test Split Completed!")

# ==========================================================
# XGBOOST PIPELINE
# ==========================================================

xgboost_pipeline = Pipeline([

    # ------------------------------------------------------
    # NLP PREPROCESSING
    # ------------------------------------------------------

    (
        "nlp",
        NLPPreprocessor()
    ),

    # ------------------------------------------------------
    # TF-IDF VECTORIZATION
    # ------------------------------------------------------

    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            max_df=0.7,
            min_df=2
        )
    ),

    # ------------------------------------------------------
    # XGBOOST MODEL
    # ------------------------------------------------------

    (
        "xgboost",
        XGBClassifier(
            n_estimators=100,
            random_state=42,
            eval_metric="logloss"
        )
    )
])

# ==========================================================
# TRAIN MODEL
# ==========================================================

print("Training XGBoost Pipeline...")

xgboost_pipeline.fit(X_train_raw, y_train)

print("Training Completed!")

# ==========================================================
# TEST MODEL
# ==========================================================

predictions = xgboost_pipeline.predict(X_test_raw)

accuracy = accuracy_score(y_test, predictions)

print("\nPipeline Accuracy:")
print(f"{accuracy:.4f}")

# ==========================================================
# SAVE MODEL
# ==========================================================

model_name = "xgboost_pipeline.pkl"

joblib.dump(xgboost_pipeline, model_name)

print("\nXGBoost Model Saved Successfully!")

# ==========================================================
# DOWNLOAD MODEL
# ==========================================================

files.download(model_name)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Train-Test Split Completed!
Training XGBoost Pipeline...


In [ ]:
# ==========================================================
# SAMPLE TEST
# ==========================================================

sample_news = """
Government announces major economic reforms to improve employment opportunities.
"""

prediction = xgboost_pipeline.predict([sample_news])[0]

if prediction == 1:
    print("\nPrediction: REAL NEWS")
else:
    print("\nPrediction: FAKE NEWS")